In [1]:
import os
import numpy as np
import pandas as pd
import networkx as nx
import itertools
import utils
import json

In [2]:
interval = 5
rep_num = 30

1. aimsun data

In [3]:
def build_aimsun_data(plan_name, overwrite=False):
    """
    Build and save unified total dataframe:
      - rep >= 0: original repetitions
      - rep = -1: mean over repetitions
    """
    total_path = f"../data/processed/{plan_name}_aimsun.parquet"

    if (not overwrite) and os.path.exists(total_path):
        print(f"Loading cached total: {plan_name}")
        return pd.read_parquet(total_path)

    print(f"Processing total: {plan_name}")

    total_all = []
    load = utils.LoadData()
    G_primal, G_dual = load.load_map_data(
        '../data/external/aimsun_topo/section_edit.shp',
        '../data/external/aimsun_topo/nodes.shp'
    )

    # 1) original repetitions
    for rep in range(rep_num):
        total = load.load_aimsun_data(plan_name, rep).copy()
        total["rep"] = rep
        total_all.append(total)

    df_total_reps = pd.concat(total_all, ignore_index=True)

    # 2) mean version -> rep = -1
    datas = (
        pd.pivot_table(
            df_total_reps,
            index=["oid", "ent"],
            values=[
                "flow", "density", "speed", "ttime",
                "CO2_interurban", "NOx_interurban",
                "VOC_interurban", "PM_interurban"
            ],
            aggfunc="mean"
        )
        .reset_index()
    )

    total_list = []
    for oid, group in datas.groupby("oid"):
        group = group.copy()
        max_flow = group["flow"].max()
        density_thres = group.loc[group["flow"] == max_flow, "density"].iloc[0]
        group["if_congest"] = (group["density"] > density_thres).astype(int)
        total_list.append(group)

    df_total_mean = pd.concat(total_list, ignore_index=True)
    df_total_mean["rep"] = -1

    # combine
    df_total = pd.concat([df_total_reps, df_total_mean], ignore_index=True)

    os.makedirs("../data/processed", exist_ok=True)
    df_total.to_parquet(total_path, index=False)

    print(f"Saved: {total_path}")

def load_aimsun(plan_name, rep=-1):
    df = pd.read_parquet(f"../data/processed/{plan_name}_aimsun.parquet")
    return df[df["rep"] == rep].copy()

In [8]:
build_aimsun_data("Random_0.1", overwrite=False)
build_aimsun_data("Random_0.5", overwrite=False)
build_aimsun_data("Random_1", overwrite=False)
build_aimsun_data("Critical_0.1", overwrite=False)
build_aimsun_data("Critical_0.5", overwrite=False)
build_aimsun_data("Critical_1", overwrite=False)
build_aimsun_data("Fix", overwrite=False)
build_aimsun_data("Actuated", overwrite=False)

Processing total: Random_0.1
Saved: ../data/processed/Random_0.1_aimsun.parquet
Processing total: Random_0.5
Saved: ../data/processed/Random_0.5_aimsun.parquet
Processing total: Random_1
Saved: ../data/processed/Random_1_aimsun.parquet
Processing total: Critical_0.1
Saved: ../data/processed/Critical_0.1_aimsun.parquet
Processing total: Critical_0.5
Saved: ../data/processed/Critical_0.5_aimsun.parquet
Processing total: Critical_1
Saved: ../data/processed/Critical_1_aimsun.parquet
Processing total: Fix
Saved: ../data/processed/Fix_aimsun.parquet
Processing total: Actuated
Saved: ../data/processed/Actuated_aimsun.parquet


In [9]:
load_aimsun("Fix", rep=-1)

,oid,speed,density,ttime,flow,ent,CO2_interurban,NOx_interurban,VOC_interurban,PM_interurban,if_congest,rep
9325800,22859,NaN,0.000000,NaN,0.0,1,0.000000,0.000000,0.000000,0.000000,0,-1
9325801,22859,NaN,0.057845,NaN,0.0,2,18.553768,0.068851,0.005141,0.012564,0,-1
9325802,22859,59.525923,1.119724,20.318035,46.0,3,283.103723,0.813882,0.138897,0.123661,0,-1
9325803,22859,58.052069,0.611158,20.860718,54.0,4,134.348098,0.470189,0.059017,0.045325,0,-1
9325804,22859,61.003422,0.420836,19.825116,30.0,5,85.337648,0.242122,0.046868,0.024848,0,-1
...,...,...,...,...,...,...,...,...,...,...,...,...
9636655,359599,49.014006,0.502791,6.260729,26.0,176,379.186358,1.747913,0.054107,0.292891,0,-1
9636656,359599,47.992001,0.097148,6.333116,6.0,177,70.665318,0.361953,0.001969,0.097010,0,-1
9636657,359599,NaN,0.000000,NaN,0.0,178,0.000000,0.000000,0.000000,0.000000,0,-1
9636658,359599,51.537454,0.154269,5.886858,8.0,179,108.358391,0.441822,0.013794,0.114424,0,-1


2. network data

In [4]:
def build_network_data(plan_name, overwrite=False):
    """
    Build and save unified timeseries dataframe from unified total dataframe.
    Expects df_total to contain:
      - rep >= 0: repetitions
      - rep = -1: mean
    """
    ts_path = f"../data/processed/{plan_name}_network.parquet"

    if (not overwrite) and os.path.exists(ts_path):
        print(f"Loading cached timeseries: {plan_name}")
        return pd.read_parquet(ts_path)

    total_path = f"../data/processed/{plan_name}_aimsun.parquet"
    if not os.path.exists(total_path):
        raise FileNotFoundError(
            f"{total_path} not found. Please run build_total_all first "
            f"or pass df_total explicitly."
        )
    df_total = pd.read_parquet(total_path)

    print(f"Processing timeseries from total: {plan_name}")

    load = utils.LoadData()
    G_primal, G_dual = load.load_map_data(
        '../data/external/aimsun_topo/section_edit.shp',
        '../data/external/aimsun_topo/nodes.shp'
    )

    def process_subgraph(G, condition):
        sub_nodes = [u for u in G.nodes if G.nodes[u].get("congest") == condition]
        subgraph = G.subgraph(sub_nodes)
        sizes = [len(c) for c in sorted(nx.connected_components(subgraph), key=len, reverse=True)]
        return subgraph, sizes

    ts_rows = []

    reps = np.sort(df_total["rep"].unique())

    for rep in reps:
        df_rep = df_total[df_total["rep"] == rep]
        ents = np.sort(df_rep["ent"].unique())

        for ent in ents:
            total_ent = df_rep[df_rep["ent"] == ent]
            G = G_dual.copy()

            for _, row in total_ent.iterrows():
                try:
                    G.nodes[int(row["oid"])].update({
                        "congest": row["if_congest"]
                    })
                except KeyError:
                    continue

            _, sizes = process_subgraph(G, condition=1)

            l_g = sizes[0] if len(sizes) > 0 else 0
            s_g = sizes[1] if len(sizes) > 1 else 0
            n_c = len(sizes)

            ts_rows.append((ent, "l_g", l_g, rep))
            ts_rows.append((ent, "n_c", n_c, rep))
            ts_rows.append((ent, "s_g", s_g, rep))

    df_ts = pd.DataFrame(ts_rows, columns=["time", "metric", "value", "rep"])

    os.makedirs("../data/processed", exist_ok=True)
    df_ts.to_parquet(ts_path, index=False)

    print(f"Saved: {ts_path}")

def load_network(plan_name, rep=-1):
    df = pd.read_parquet(f"../data/processed/{plan_name}_network.parquet")
    return df[df["rep"] == rep].copy()

In [11]:
build_network_data("Random_0.1", overwrite=False)
build_network_data("Random_0.5", overwrite=False)
build_network_data("Random_1", overwrite=False)
build_network_data("Critical_0.1", overwrite=False)
build_network_data("Critical_0.5", overwrite=False)
build_network_data("Critical_1", overwrite=False)
build_network_data("Fix", overwrite=False)
build_network_data("Actuated", overwrite=False)

Processing timeseries from total: Random_0.1
Saved: ../data/processed/Random_0.1_network.parquet
Processing timeseries from total: Random_0.5
Saved: ../data/processed/Random_0.5_network.parquet
Processing timeseries from total: Random_1
Saved: ../data/processed/Random_1_network.parquet
Processing timeseries from total: Critical_0.1
Saved: ../data/processed/Critical_0.1_network.parquet
Processing timeseries from total: Critical_0.5
Saved: ../data/processed/Critical_0.5_network.parquet
Processing timeseries from total: Critical_1
Saved: ../data/processed/Critical_1_network.parquet
Processing timeseries from total: Fix
Saved: ../data/processed/Fix_network.parquet
Processing timeseries from total: Actuated
Saved: ../data/processed/Actuated_network.parquet


In [5]:
load_network("Fix", rep=-1)

,time,metric,value,rep
0,1,l_g,1,-1
1,1,n_c,2,-1
2,1,s_g,1,-1
3,2,l_g,2,-1
4,2,n_c,9,-1
...,...,...,...,...
535,179,n_c,2,-1
536,179,s_g,1,-1
537,180,l_g,1302,-1
538,180,n_c,3,-1


3. critical time and location

In [21]:
def build_critical_timeloc(plan_name, threshold=0.9, overwrite=False):
    """
    Save critical metrics for:
      - rep = -1 : mean version
      - rep >= 0 : Monte Carlo repetitions

    Output:
      1. {plan_name}_critical_summary.parquet
      2. {plan_name}_critical_vector.parquet
    """

    summary_path = f"../data/processed/{plan_name}_critical_summary.parquet"
    vector_path = f"../data/processed/{plan_name}_critical_vector.parquet"

    if (not overwrite) and os.path.exists(summary_path) and os.path.exists(vector_path):
        print(f"Loading cached: {plan_name}")
        df_summary = pd.read_parquet(summary_path)
        df_vector = pd.read_parquet(vector_path)
        return df_summary, df_vector

    print(f"Processing critical metrics for mean + all reps: {plan_name}")

    load = utils.LoadData()
    G_primal, G_dual = load.load_map_data(
        '../data/external/aimsun_topo/section_edit.shp',
        '../data/external/aimsun_topo/nodes.shp'
    )

    summary_rows = []
    vector_rows = []

    # =========================
    # 1) mean version -> rep = -1
    # =========================
    df_total = pd.read_parquet(f"../data/processed/{plan_name}_aimsun.parquet")
    mean_total = df_total[df_total["rep"] == -1]

    df_ts = pd.read_parquet(f"../data/processed/{plan_name}_network.parquet")
    median_ts = (
        df_ts[
            (df_ts["rep"].between(0, 29))
            & (df_ts["metric"].isin(["l_g", "s_g"]))
        ]
        .groupby(["time", "metric"], as_index=False)["value"]
        .median()
    )

    l_g = (
        median_ts[median_ts["metric"] == "l_g"]
        .sort_values("time")["value"]
        .to_numpy()
    )

    s_g = (
        median_ts[median_ts["metric"] == "s_g"]
        .sort_values("time")["value"]
        .to_numpy()
    )

    critical_time = load.calculate_percolation_time(
        l_g, s_g, interval=60, method="l"
    )


    critical_loc, intersection = load.calculate_critical_locations(
        mean_total, critical_time, threshold
    )

    summary_rows.append({
        "plan": plan_name,
        "rep": -1,
        "critical_time": critical_time,
        "threshold": threshold,
        "n_critical_loc": len(critical_loc),
        "n_intersection": len(intersection),
    })

    for i, v in enumerate(critical_loc):
        vector_rows.append({
            "plan": plan_name,
            "rep": -1,
            "index": i,
            "type": "critical_loc",
            "value": v
        })

    for i, v in enumerate(intersection):
        vector_rows.append({
            "plan": plan_name,
            "rep": -1,
            "index": i,
            "type": "intersection",
            "value": v
        })

    # =========================
    # 2) all repetitions
    # =========================
    df_total = pd.read_parquet(f"../data/processed/{plan_name}_aimsun.parquet")
    total = df_total[df_total["rep"] != -1]

    df_ts = pd.read_parquet(f"../data/processed/{plan_name}_network.parquet")
    ts = df_ts[df_ts["rep"] != -1]

    reps = np.sort(ts["rep"].unique())

    for rep in reps:
        print(f"  rep = {rep}")

        ts_rep = ts.loc[ts["rep"] == rep].copy()
        total_rep = total.loc[total["rep"] == rep].copy()

        l_g = ts_rep.loc[ts_rep["metric"] == "l_g", "value"].to_numpy()
        s_g = ts_rep.loc[ts_rep["metric"] == "s_g", "value"].to_numpy()

        critical_time = load.calculate_percolation_time(
            l_g, s_g, interval=60, method="l"
        )

        critical_loc, intersection = load.calculate_critical_locations(
            total_rep, critical_time, threshold
        )

        summary_rows.append({
            "plan": plan_name,
            "rep": rep,
            "critical_time": critical_time,
            "threshold": threshold,
            "n_critical_loc": len(critical_loc),
            "n_intersection": len(intersection),
        })

        for i, v in enumerate(critical_loc):
            vector_rows.append({
                "plan": plan_name,
                "rep": rep,
                "index": i,
                "type": "critical_loc",
                "value": v
            })

        for i, v in enumerate(intersection):
            vector_rows.append({
                "plan": plan_name,
                "rep": rep,
                "index": i,
                "type": "intersection",
                "value": v
            })

    df_summary = pd.DataFrame(summary_rows)
    df_vector = pd.DataFrame(vector_rows)

    os.makedirs("../data/processed", exist_ok=True)
    df_summary.to_parquet(summary_path, index=False)
    df_vector.to_parquet(vector_path, index=False)

    print(f"Saved: {summary_path}")
    print(f"Saved: {vector_path}")

def load_critical(plan_name, rep=-1):
    df_summary = pd.read_parquet(f"../data/processed/{plan_name}_critical_summary.parquet")
    df_vector = pd.read_parquet(f"../data/processed/{plan_name}_critical_vector.parquet")

    critical_time = df_summary.loc[df_summary["rep"] == rep, "critical_time"].iloc[0]

    critical_loc = df_vector.loc[
        (df_vector["rep"] == rep) & (df_vector["type"] == "critical_loc"),
        "value"
    ].to_numpy()

    intersection = df_vector.loc[
        (df_vector["rep"] == rep) & (df_vector["type"] == "intersection"),
        "value"
    ].to_numpy()

    return critical_time, critical_loc, intersection

In [22]:
build_critical_timeloc("Fix", threshold=0.9, overwrite=True)

Processing critical metrics for mean + all reps: Fix
  rep = 0
  rep = 1
  rep = 2
  rep = 3
  rep = 4
  rep = 5
  rep = 6
  rep = 7
  rep = 8
  rep = 9
  rep = 10
  rep = 11
  rep = 12
  rep = 13
  rep = 14
  rep = 15
  rep = 16
  rep = 17
  rep = 18
  rep = 19
  rep = 20
  rep = 21
  rep = 22
  rep = 23
  rep = 24
  rep = 25
  rep = 26
  rep = 27
  rep = 28
  rep = 29
Saved: ../data/processed/Fix_critical_summary.parquet
Saved: ../data/processed/Fix_critical_vector.parquet


In [ ]:
critical_time_mean, critical_loc_mean, intersection_mean = load_critical("Fix", rep=-1)
critical_time_0, critical_loc_0, intersection_0 = load_critical("Fix", rep=0)

np.int64(57)

4. threshold

In [17]:
load = utils.LoadData()
G_primal, G_dual = load.load_map_data(
    '../data/external/aimsun_topo/section_edit.shp',
    '../data/external/aimsun_topo/nodes.shp'
)

plan_name = "Fix"
df_total = pd.read_parquet(f"../data/processed/{plan_name}_aimsun.parquet")
mean_total = df_total[df_total["rep"] == -1]
critical_time, critical_loc_mean, intersection_mean = load_critical(plan_name, rep=-1)

thresholds = np.arange(0.01, 1.01, 0.01)
res = []
for threshold in thresholds:
    critical_loc, intersection = load.calculate_critical_locations(
        mean_total, critical_time, threshold
    )
    intersection = [
        [u, v] for u, v, k, e in G_primal.edges(data=True, keys=True)
        if e['name'] in critical_loc
    ]
    intersection = set(
        item for sublist in intersection for item in sublist
        if 'd' not in str(item) and 'e' not in str(item) and 's' not in str(item)
    )
    res.append({
        "threshold": threshold,
        "num_intersections": len(intersection)
    })
df_res = pd.DataFrame(res)
df_res.to_parquet(f"../data/processed/{plan_name}_critical_scaling.parquet")